In [ ]:
# Practical Deep Learning for Coders 2022
import matplotlib.pyplot as plt
import pandas as pd
from fastai.data.external import *

# 下载 MNIST 采样集（只含 3 和 7 两类手写数字）
path = untar_data(URLs.MNIST_SAMPLE)


In [ ]:
# 设置显示时的基准路径，让后续 ls() 打印相对路径更简洁
# 注意：BASE_PATH 是 Path 类的属性，要设在类上，不能设在实例 path 上
Path.BASE_PATH = path   # 修正：原来是 path.BASE_PATH=path，会报 AttributeError


In [ ]:
# 查看数据集目录结构
path.ls()


In [ ]:
# 分别取训练集里所有的 "3" 和 "7" 图片路径并排序
threes = (path/'train'/'3').ls().sorted()
sevens = (path/'train'/'7').ls().sorted()
threes


In [ ]:
# 打开其中一张 "3" 的图片看看（用 PIL 的 Image.open）
from PIL import Image

im3_path = threes[1]
im3 = Image.open(im3_path)   # 修正：原来是 gr.Image.open，Gradio 没有 open 方法
im3


In [ ]:
# 把图片转成 NumPy 数组，看左上角一小块的像素值
import numpy as np

np.array(im3)[4:10, 4:10]   # 修正：原来用了 python 的 array 模块，无法二维切片，应为 np.array


In [ ]:
# 同样地，转成 PyTorch 张量看像素值
from torch import tensor

tensor(im3)[4:10, 4:10]


In [ ]:
# 用 DataFrame 更直观地展示一块像素矩阵
im3_t = tensor(im3)
df = pd.DataFrame(im3_t[4:15, 4:22])
df


In [ ]:
# —— 用预训练模型（timm 的 convnext）做推理 ——
from fastai.vision.all import *
import gradio as gr
import timm


In [ ]:
# 载入一张测试图（篮子犬 basset），生成缩略图
im = PILImage.create('basset.jpg')
im.thumbnail((224, 224))
im


In [ ]:
# 载入导出的模型并预测
learn = load_learner('model.pkl')
learn.predict(im)


In [ ]:
# （备注）训练该模型时用的配置，供参考：
# learn = vision_learner(dls, 'convnext_tiny_in22k', metrics=error_rate).to_fp16()
# learn.fine_tune(3)              # 注意：原注释里把 fine_tune 写成了 find_tune
# timm.list_models("convnext")   # 列出所有 convnext 变体


In [ ]:
# 用模型词表作类别，把预测概率整理成字典
categories = learn.dls.vocab
def classify_image(img):
    pred, idx, probs = learn.predict(img)
    return dict(zip(categories, map(float, probs)))
classify_image(im)


In [ ]:
# 定义 Gradio 的输入(图片)/输出(标签)组件（Gradio 3.x 旧写法）
image = gr.inputs.Image(shape=(192, 192))
label = gr.outputs.Label()
examples = ['basset.jpg']


In [ ]:
# 组装 Gradio 界面
intf = gr.Interface(fn=classify_image, inputs=image, outputs=label, examples=examples)


In [ ]:
# 启动界面（inline=False 表示不在 notebook 内嵌显示）
intf.launch(inline=False)


In [ ]:
# 查看模型结构
m = learn.model
m


In [ ]:
# 取出某个子模块（stem 的第 1 层）看它的参数
l = m.get_submodule('0.model.stem.1')
list(l.parameters())


In [ ]:
# 取出更深处一个全连接层的参数
l = m.get_submodule("0.model.stages.0.blocks.1.mlp.fc1")
list(l.parameters())


In [ ]:
# 一个二次函数示例
def f(x): return 3*x**2 + 2*x + 1   # 修正：原来是 3*x*2（其实等于 6x），二次项应为 3*x**2


In [ ]:
# —— 以下辅助函数原课程有定义、但本 notebook 里漏了，补上以免后面 cell 报未定义错误 ——
from functools import partial
import torch

# 通用二次函数： a*x^2 + b*x + c
def quad(a, b, c, x): return a*x**2 + b*x + c

# 固定 a,b,c，返回一个只需传 x 的二次函数（方便传给绘图/交互）
def mk_quad(a, b, c): return partial(quad, a, b, c)

# 修正线性单元 ReLU：先算 m*x+b，再把负数截断为 0
def rectified_linear(m, b, x):
    y = m*x + b
    return torch.clip(y, 0.)

# 通用绘图工具：把任意函数 f 画出来，方便和散点数据对比
def plot_function(f, title=None, min=-2.1, max=2.1, color='r', ylim=None):
    x = torch.linspace(min, max, 100)[:, None]
    if ylim: plt.ylim(ylim)
    plt.plot(x, f(x), color)
    if title is not None: plt.title(title)


In [ ]:
# 生成带噪声的样本数据，模拟真实观测
from numpy.random import normal, seed, uniform
np.random.seed(42)
def noise(x, scale): return normal(scale=scale, size=x.shape)
def add_noise(x, mult, add): return x*(1 + noise(x, mult)) + noise(x, add)


In [ ]:
# 在 [-2,2] 取 20 个点，套上二次函数并加噪声，画散点图
x = torch.linspace(-2, 2, steps=20)[:, None]
y = add_noise(f(x), 0.3, 1.5)
plt.scatter(x, y)


In [ ]:
# 交互式滑块：手动调 a,b,c，观察二次曲线如何拟合散点
from ipywidgets import interact
@interact(a=1.5, b=1.5, c=1.5)
def plot_quad(a, b, c):
    plt.scatter(x, y)
    plot_function(mk_quad(a, b, c), ylim=(-3, 12))


In [ ]:
# 两个 ReLU 相加：这是神经网络能拟合任意曲线的最小"积木"
def double_relu(m1, b1, m2, b2, x):
    return rectified_linear(m1, b1, x) + rectified_linear(m2, b2, x)


In [ ]:
# 交互式滑块：调 4 个参数，看两个 ReLU 叠加出的折线形状
@interact(m1=-1.5, b1=-1.5, m2=1.5, b2=1.5)   # 修正：原来 b1=-1,5 是逗号，会导致 SyntaxError
def plot_double_relu(m1, b1, m2, b2):
    plot_function(partial(double_relu, m1, b1, m2, b2), ylim=(-1, 6))
